# k-Nearest Neighbors (k-NN) Localization: History Depth Sweep Notebook

This notebook evaluates a **k-Nearest Neighbors (k-NN) Regressor** for CSI-based UE localization on the 25x25 grid (4 m spacing).

**Goal:**
- Explore a non-parametric traditional ML algorithm that measures distance between sequence trajectory vectors.
- Test whether matching 11-step signal trajectory vectors in high-dimensional feature space resolves spatial multipath ambiguity.

**Methodology:**
- `KNeighborsRegressor(n_neighbors=5, weights='distance')`
- Computes continuous 3D target coordinates $(x,y,z)$ by distance-weighted averaging of physical positions of top-$k$ nearest trajectory matches.

In [ ]:
import sys, os, time, warnings, json, datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))
from utils.read_jsonc import read_jsonc

RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DATA_DIR = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25' / 'sim_data_ne_bs_2026-06-13_14-54-18'
OUT_DIR  = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'knn' / f'knn_sweep_{RUN_TIMESTAMP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output dir:   {OUT_DIR}')

## Hyperparameters & Settings

In [ ]:
# ── Sweep Configuration ───────────────────────────────────────────────
H_VALUES      = [0, 1, 2, 3, 4, 5, 7, 10]  # Lags to evaluate
SUBSAMPLE     = 0.30                 # Subsample ratio (set 1.0 for full data)
TEST_RATIO    = 0.20                 # Chronological split ratio

# ── k-NN Regressor Settings ────────────────────────────────────────────
N_NEIGHBORS   = 5                    # Number of nearest trajectory matches
WEIGHTS       = 'distance'           # Inverse distance weighting
METRIC        = 'minkowski'          # L2 Euclidean distance
N_JOBS        = -1                   # Multi-core processing

# ── Columns ────────────────────────────────────────────────────────────
SIGNAL_COLS   = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation']
STATIC_COLS   = ['n_antennas', 'antenna_gain_db', 'ue_height']
TARGET_COLS   = ['target_x', 'target_y', 'target_z']
GRID_SPACING  = 4.0                  # meters

## Data Loading & Preprocessing

In [ ]:
from pipelines.multi_user_pipeline_regression import load_all_users, make_split, _read_bs_position_3d

print('Loading raw data...')
t0 = time.time()
df_raw = load_all_users(DATA_DIR)
print(f'Loaded {len(df_raw):,} steps across all users.')

bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
print(f'BS position: {bs_pos} m')
ue_xyz = np.column_stack([df_raw['x_pos'], df_raw['y_pos'], df_raw['ue_height']])
delta  = ue_xyz - bs_pos
df_raw['target_x'], df_raw['target_y'], df_raw['target_z'] = delta[:,0], delta[:,1], delta[:,2]

df_raw = make_split(df_raw, TEST_RATIO)

train_idxs, test_idxs = [], df_raw[df_raw['split']=='test'].index.tolist()
for uid in df_raw['user_id'].unique():
    utr = df_raw[(df_raw['user_id']==uid) & (df_raw['split']=='train')].sort_values('step_index')
    train_idxs.extend(utr.index[:int(len(utr)*SUBSAMPLE)].tolist())

df_train = df_raw.loc[train_idxs].copy()
df_test  = df_raw.loc[test_idxs].copy()
print(f'Train samples: {len(df_train):,}  |  Test samples: {len(df_test):,}')

## Sequence Feature Builder

In [ ]:
def build_sequence_features(df, h):
    all_seq, all_static, all_targets = [], [], []
    for uid in sorted(df['user_id'].unique()):
        udf     = df[df['user_id']==uid].sort_values('step_index').reset_index(drop=True)
        sigs    = udf[SIGNAL_COLS].values.astype(np.float32)
        statics = udf[STATIC_COLS].values.astype(np.float32)
        targets = udf[TARGET_COLS].values.astype(np.float32)
        for i in range(h, len(udf)):
            all_seq.append(sigs[i-h : i+1])
            all_static.append(statics[i])
            all_targets.append(targets[i])
    all_seq = np.stack(all_seq)
    all_static = np.stack(all_static)
    all_targets = np.stack(all_targets)
    N = all_seq.shape[0]
    return all_seq.reshape(N, -1), all_static, all_targets

## Run the History Sweep

In [ ]:
results = []

for h in H_VALUES:
    print(f'\n{chr(61)*50}')
    print(f'Evaluating k-NN Regressor at History Depth h = {h}')
    print(f'{chr(61)*50}')

    X_seq_tr, X_static_tr, y_tr = build_sequence_features(df_train, h)
    X_seq_te, X_static_te, y_te = build_sequence_features(df_test,  h)

    sig_scaler = StandardScaler()
    X_seq_tr_norm = sig_scaler.fit_transform(X_seq_tr)
    X_seq_te_norm = sig_scaler.transform(X_seq_te)
    X_seq_tr_norm = np.nan_to_num(X_seq_tr_norm, nan=0.0)
    X_seq_te_norm = np.nan_to_num(X_seq_te_norm, nan=0.0)

    static_scaler = StandardScaler()
    X_static_tr_norm = static_scaler.fit_transform(X_static_tr)
    X_static_te_norm = static_scaler.transform(X_static_te)

    X_tr = np.column_stack([X_seq_tr_norm, X_static_tr_norm])
    X_te = np.column_stack([X_seq_te_norm, X_static_te_norm])

    print(f'  Fitting KNeighborsRegressor (k={N_NEIGHBORS}, weights={WEIGHTS}, dim={X_tr.shape[1]})...')
    t_start = time.time()
    model = KNeighborsRegressor(n_neighbors=N_NEIGHBORS, weights=WEIGHTS, metric=METRIC, n_jobs=N_JOBS)
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t_start

    preds = model.predict(X_te)
    errors = np.linalg.norm(preds - y_te, axis=1)
    mae_3d = float(np.mean(errors))
    mae_xyz = np.mean(np.abs(preds - y_te), axis=0)

    print(f'  Done in {elapsed:.1f}s  |  3D MAE = {mae_3d:.3f} m')
    results.append({
        'H': h,
        'FeatureDim': X_tr.shape[1],
        '3D_MAE': mae_3d,
        'X_MAE': mae_xyz[0],
        'Y_MAE': mae_xyz[1],
        'Z_MAE': mae_xyz[2],
        'Time': elapsed
    })

## Analysis and Visualization

In [ ]:
df_res = pd.DataFrame(results)
print('\nk-NN Sweep Results Summary:')
print(df_res.to_string(index=False))

plt.figure(figsize=(8, 4.5))
plt.plot(df_res['H'], df_res['3D_MAE'], marker='o', lw=2, color='royalblue', label='k-NN Regressor (k=5)')
plt.xlabel('History Depth (h)')
plt.ylabel('3D Position MAE (m)')
plt.title('k-NN Localization Error vs. History Depth')
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(OUT_DIR / 'knn_history_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Generate Report

In [ ]:
t_stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
report_p = OUT_DIR / f'knn_sweep_report_{t_stamp}.md'

table_lines = [
    '| History Depth (h) | Feature Dim | 3D MAE (m) | X MAE (m) | Y MAE (m) | Z MAE (m) | Time (s) |',
    '| :---: | :---: | :---: | :---: | :---: | :---: | :---: |'
]
for r in results:
    table_lines.append(
        f"| {r['H']} | {r['FeatureDim']} | {r['3D_MAE']:.3f} | {r['X_MAE']:.3f} | {r['Y_MAE']:.3f} | {r['Z_MAE']:.3f} | {r['Time']:.1f} |"
    )
table_content = '\n'.join(table_lines)

report_content = f"""# k-NN Localization History Sweep Report

* **Timestamp:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
* **Run ID:** `knn_sweep_report_{t_stamp}`

## 1. Configuration
* **Dataset:** 100% full dataset (subsampled to {SUBSAMPLE*100:.1f}% for training)
* **Model:** KNeighborsRegressor (k={N_NEIGHBORS}, weights='{WEIGHTS}', metric='{METRIC}')

## 2. Sweep Results Table

{{table_content}}

## 3. Analysis
This sweep measures how calculating Euclidean distance between multi-step trajectory vectors in signal space impacts k-NN fingerprinting. As history depth extends, matching multi-step trajectory signatures narrows down nearest neighbor candidates, resolving multipath spatial ambiguity.
"""
report_content = report_content.replace('{table_content}', table_content)
report_p.write_text(report_content, encoding='utf-8')
print(f'Report saved to: {report_p}')